In [1]:
# test_servers.py
import time
from pathlib import Path
import servers
import warnings
from sklearn.exceptions import InconsistentVersionWarning
warnings.filterwarnings("ignore", category=InconsistentVersionWarning)

SERVER_INDEX = 1         # which server to instantiate
USE_DO_SLEEP = False     # set True if you want the call to block for the processing time

class SimpleRequest:
    def __init__(self, combination, message_size=1000, bandwidth=1000.0, load=None):
        """
        process_id: integer row index used as fallback (keep as in CSV indexing)
        combination: single-letter string ('s','p','d') - used by Server for combined logic
        """
        self.combination = combination  # NEW: single-letter combination attribute
        self.message_size = message_size
        self.bandwidth = bandwidth
        self.load = load if load is not None else {}

    def __repr__(self):
        return f"SimpleRequest(, comb='{self.combination}')"

print("current time:", time.time())
srv = servers.Server(SERVER_INDEX)

# Compose 4 requests: combinations s, p, d, p
# process_id values mirror your CSV row selections (these were: 2, 15, 16, 19 in original example)
# In your earlier example you subtracted 2 (2-2, 15-2, ...). Keep the same indices here.
req1 = SimpleRequest( combination='s', message_size=2000, bandwidth=1.0, load={SERVER_INDEX:1.0})
req2 = SimpleRequest(combination='p', message_size=2000, bandwidth=1.0, load={SERVER_INDEX:1.0})
req3 = SimpleRequest(combination='d', message_size=2000, bandwidth=1.0, load={SERVER_INDEX:1.0})
req4 = SimpleRequest(combination='p', message_size=2000, bandwidth=1.0, load={SERVER_INDEX:1.0})

print("\nScheduling 4 requests sequentially with combos: s, p, d, p\n")

res = srv.schedule_request(req1, do_sleep=USE_DO_SLEEP)
print(f"req1 {req1} -> accepted={res[0]}, finish_or_reason={res[1]}, total_delay={res[2]:.6f}")

res = srv.schedule_request(req2, do_sleep=USE_DO_SLEEP)
print(f"req2 {req2} -> accepted={res[0]}, finish_or_reason={res[1]}, total_delay={res[2]:.6f}")

res = srv.schedule_request(req3, do_sleep=USE_DO_SLEEP)
print(f"req3 {req3} -> accepted={res[0]}, finish_or_reason={res[1]}, total_delay={res[2]:.6f}")

res = srv.schedule_request(req4, do_sleep=USE_DO_SLEEP)
print(f"req4 {req4} -> accepted={res[0]}, finish_or_reason={res[1]}, total_delay={res[2]:.6f}")

print("\n---------------------------")
srv.print_active_requests()
print("---------------------------\n")

print("Scheduling 5th request")
req5 = SimpleRequest(combination='s', message_size=2000, bandwidth=10.0, load={SERVER_INDEX:1.0})
res5 = srv.schedule_request(req5, do_sleep=USE_DO_SLEEP)
print(f"req5 {req5} -> accepted={res5[0]}, finish_or_reason={res5[1]}, total_delay={res5[2]:.6f}")

# Advance time to free one slot: pick earliest finish_time and jump a tiny bit after it
if srv.active_requests:
    earliest_finish = min(ar['finish_time'] for ar in srv.active_requests)
    new_time = earliest_finish + 1e-6
    freed = srv.update_active_requests(current_time=new_time)
    print(f"\nAdvanced time to {new_time:.6f}; freed slots: {freed}; srv.num_requests now: {getattr(srv, 'num_requests', 'N/A')}\n")
else:
    print("\nNo active requests to advance.\n")

# Retry scheduling req5 if it previously failed
if not res5[0]:
    res5_retry = srv.schedule_request(req5, current_time=new_time, do_sleep=USE_DO_SLEEP)
    print(f"Retry req5 -> accepted={res5_retry[0]}, finish_or_reason={res5_retry[1]}, total_delay={res5_retry[2]:.6f}")

print("\nFinal active requests:")
print("---------------------------")
srv.print_active_requests()
print("---------------------------\n")



current time: 1762872173.6592832

Scheduling 4 requests sequentially with combos: s, p, d, p


[SERVER]    Using predictor for letter 's' with combined string 's'

req1 SimpleRequest(, comb='s') -> accepted=True, finish_or_reason=1762872173.97076, total_delay=0.099628

[SERVER]    Using predictor for letter 'p' with combined string 'ps'

req2 SimpleRequest(, comb='p') -> accepted=True, finish_or_reason=1762872173.9788682, total_delay=0.105522

[SERVER]    Using predictor for letter 'd' with combined string 'dps'

req3 SimpleRequest(, comb='d') -> accepted=True, finish_or_reason=1762872173.9626555, total_delay=0.081445

[SERVER]    Using predictor for letter 'p' with combined string 'pdps'

req4 SimpleRequest(, comb='p') -> accepted=True, finish_or_reason=1762872173.9613495, total_delay=0.078022

---------------------------
Server 1 Active Requests:
  Start: 1762872173.87, Finish: 1762872173.97, Proc Time: 0.099628
  Start: 1762872173.87, Finish: 1762872173.98, Proc Time: 0.105522
  Sta

c:\Users\Shrutya\miniconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
c:\Users\Shrutya\miniconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
